In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from evaluation_src.scores_evaluator import ScoresEvaluator
from clans3d.core.input_file_type import InputFileType
from clans3d.similarity.tool_type import ToolType
import pandas as pd

In [ ]:
WORKING_DIR = "../work"
EXAMPLE_DATASET_FASTA = "../../examples/small_fasta_files/50.fasta"
EXAMPLE_DATASET_TSV = "../../examples/big_tsv_files/1500.tsv"

In [ ]:
evaluator = ScoresEvaluator(WORKING_DIR)

In [ ]:
# Generate clans files from example dataset
clans_files = evaluator.generate_clans_files(EXAMPLE_DATASET_FASTA, InputFileType.FASTA, ToolType.FOLDSEEK, "evalue")
print(clans_files)

In [ ]:
# Parameters for clustering with recovered clans.jar The first entry in each tuple is for structure-based clans, the second for sequence-based clans.
PATH_TO_CLANS_JAR = "../../src/clans3d/legacy/clans_working_version.jar" # path to clans jar file
CLUSTER_ROUNDS: tuple[int, int] = (1000, 1000) # sets the number of rounds for clustering in recovered clans
P_VALUES: tuple[float, float] = (1E-10, 1E-10) # sets the p-value threshold for clustering in recovered clans for (structure, sequence)
CLUSTER_2D: tuple[bool, bool] = (False, False) # whether to use 2D clustering in recovered clans
VERBOSE: bool = True # whether to print verbose output from recovered clans

In [ ]:
# Cluster the generated clans files
clustered_clans_files = evaluator.cluster_clans_files(PATH_TO_CLANS_JAR, clans_files, CLUSTER_ROUNDS, P_VALUES, CLUSTER_2D, VERBOSE)
print(clustered_clans_files)

In [ ]:
# from here use already clustered clans files for evaluation
evaluator = ScoresEvaluator(WORKING_DIR)
clustered_clans_files = ("../../examples/clans_files/500_cleaned_clustered_r_10000_p_1e-10.clans",
                         "../../examples/clans_files/500_cleaned_seq_clustered_r_10000_p_1e-29.clans")

In [ ]:
# Extract data from clustered CLANS files
df_scores, df_euclidean_dist, df_coord = evaluator.extract_data_from_clans_files(clustered_clans_files)
display(df_scores.head())
display(df_euclidean_dist.head())
display(df_coord.head())

In [ ]:
# Generate scatter plots to visualize relationships between scores and euclidian-distances
evaluator.visualizer.generate_scatter_plot(
    data_x=df_scores["Score_-log10_struct"],
    data_y=df_scores["Score_-log10_seq"],
    x_label="Structural Score (-log10)",
    y_label="Sequence Score (-log10)",
    title="Structural vs Sequence Scores"
)

evaluator.visualizer.generate_scatter_plot(
    data_x=df_euclidean_dist["euclidean_dist_struct"],
    data_y=df_euclidean_dist["euclidean_dist_seq"],
    x_label="Structural Euclidean Distance",
    y_label="Sequence Euclidean Distance",
    title="Structural vs Sequence Euclidean Distances"
)

### Interpretation of plots

1: kind of linear but sequence based evalues seem a lot more optimistic than structure based evalues

2: bands show that structural distance seems to be very different whereas sequence distance stays similar

In [ ]:
# Find clusters using density-based and graph-based methods
RESOLUTION_PARAMS = (1.007, 1.01)  # (structure, sequence)
df_cluster_labels = evaluator.clustering.find_clusters_density_based(df_coord, "struct")
df_cluster_labels = df_cluster_labels.merge(evaluator.clustering.find_clusters_density_based(df_coord, "seq"), on="Sequence_ID")
df_Leiden_labels = evaluator.clustering.find_clusters_graph_based(df_scores, "Score_-log10_struct", resolution=RESOLUTION_PARAMS[0])
df_Leiden_labels = df_Leiden_labels.merge(evaluator.clustering.find_clusters_graph_based(df_scores, "Score_-log10_seq", resolution=RESOLUTION_PARAMS[1]), on="Sequence_ID")
df_cluster_labels = df_cluster_labels.merge(df_Leiden_labels, on="Sequence_ID")
display(df_cluster_labels)

In [ ]:
# Display number of clusters and noise found by each method

struct_hdbscan_counts = evaluator.clustering.get_cluster_counts(df_cluster_labels, "cluster_id_struct_HDBSCAN")
seq_hdbscan_counts = evaluator.clustering.get_cluster_counts(df_cluster_labels, "cluster_id_seq_HDBSCAN")
struct_leiden_counts = evaluator.clustering.get_cluster_counts(df_cluster_labels, "cluster_id_Score_-log10_struct_Leiden")
seq_leiden_counts = evaluator.clustering.get_cluster_counts(df_cluster_labels, "cluster_id_Score_-log10_seq_Leiden")

# expected number of clusters in clans file (500) struct visually 4 - 5
# expected number of clusters in clans file (500) seq visually 3
data_for_plot = {
    'HDBSCAN (Struct)': struct_hdbscan_counts,
    'HDBSCAN (Seq)': seq_hdbscan_counts,
    'Leiden (Struct)': struct_leiden_counts,
    'Leiden (Seq)': seq_leiden_counts
}
df_found_clusters = pd.DataFrame(data_for_plot)
display(df_found_clusters)

### Interpretation

The clustering of the structural file seems to have produced more clusters than the clustering on the sequence file.

Density and network based methods seem to agree on clustered sequence file

In [ ]:
# Compute ARI and NMI between structure-based and sequence-based clusterings
clustering_agreement_HDBSCAN = evaluator.clustering.compute_clustering_agreement(df_cluster_labels, "cluster_id_struct_HDBSCAN", "cluster_id_seq_HDBSCAN")
clustering_agreement_Leiden = evaluator.clustering.compute_clustering_agreement(df_cluster_labels, "cluster_id_Score_-log10_struct_Leiden", "cluster_id_Score_-log10_seq_Leiden")
print(f"Clustering agreement (HDBSCAN): ARI={clustering_agreement_HDBSCAN['ARI']}, NMI={clustering_agreement_HDBSCAN['NMI']}")
print(f"Clustering agreement (Leiden): ARI={clustering_agreement_Leiden['ARI']}, NMI={clustering_agreement_Leiden['NMI']}")

### Interpretation

ari and nmi with hdbscan very low while very high with network based clustering?

weird

Maybe because hdbscan found a lot more clusters then the network based method

In [ ]:
# Compute Jaccard overlap between structure-based and sequence-based clusterings
jaccard_results_HDBSCAN = evaluator.clustering.compute_Jaccard_overlap(df_cluster_labels, "cluster_id_struct_HDBSCAN", "cluster_id_seq_HDBSCAN", remove_zero_jaccard=False)
jaccard_results_Leiden = evaluator.clustering.compute_Jaccard_overlap(df_cluster_labels, "cluster_id_Score_-log10_struct_Leiden", "cluster_id_Score_-log10_seq_Leiden", remove_zero_jaccard=False)
print("Jaccard Overlap Results (HDBSCAN - structure vs sequence):")
display(jaccard_results_HDBSCAN)
print("Jaccard Overlap Results (Leiden - structure vs sequence):")
display(jaccard_results_Leiden)


In [ ]:
# Heatmap for Jaccard indices from HDBSCAN clustering
jaccard_pivot_hdb = jaccard_results_HDBSCAN.pivot(
    index='cluster_id_struct_HDBSCAN',
    columns='cluster_id_seq_HDBSCAN',
    values='JaccardIndex'
)
evaluator.visualizer.generate_heatmap(
    pivot_table=jaccard_pivot_hdb,
    title='Jaccard Index Heatmap (HDBSCAN: Structure vs. Sequence)',
    x_label='Sequence Clusters',
    y_label='Structure Clusters'
)

# Heatmap for Jaccard indices from Leiden clustering
jaccard_pivot_l = jaccard_results_Leiden.pivot(
    index='cluster_id_Score_-log10_struct_Leiden',
    columns='cluster_id_Score_-log10_seq_Leiden',
    values='JaccardIndex'
)
evaluator.visualizer.generate_heatmap(
    pivot_table=jaccard_pivot_l,
    title='Jaccard Index Heatmap (Leiden: Structure vs. Sequence)',
    x_label='Sequence Clusters',
    y_label='Structure Clusters'
)

### Interpretation

The Jaccard index should show me there are structural clusters very similar to sequence clusters.

In [ ]:
# Compute overlap coefficient between structure-based and sequence-based clusterings
overlap_coefficient_HDBSCAN = evaluator.clustering.compute_overlap_coefficient(df_cluster_labels, "cluster_id_struct_HDBSCAN", "cluster_id_seq_HDBSCAN", drop_zero=False)
overlap_coefficient_Leiden = evaluator.clustering.compute_overlap_coefficient(df_cluster_labels, "cluster_id_Score_-log10_struct_Leiden", "cluster_id_Score_-log10_seq_Leiden", drop_zero=False)
print("Overlap Coefficient Results (HDBSCAN - structure vs sequence):")
display(overlap_coefficient_HDBSCAN)
print("Overlap Coefficient Results (Leiden - structure vs sequence):")
display(overlap_coefficient_Leiden)

In [ ]:
# Heatmap for Overlap coefficients HDBSCAN
overlap_pivot = overlap_coefficient_HDBSCAN.pivot(
    index='cluster_id_struct_HDBSCAN',
    columns='cluster_id_seq_HDBSCAN',
    values='OverlapCoefficient'
)
evaluator.visualizer.generate_heatmap(
    pivot_table=overlap_pivot,
    title='Overlap Coefficient Heatmap (HDBSCAN: Structure vs. Sequence)',
    x_label='Sequence Clusters',
    y_label='Structure Clusters'
)

# Heatmap for Overlap coefficients Leiden
overlap_pivot = overlap_coefficient_Leiden.pivot(
    index='cluster_id_Score_-log10_struct_Leiden',
    columns='cluster_id_Score_-log10_seq_Leiden',
    values='OverlapCoefficient'
)
evaluator.visualizer.generate_heatmap(
    pivot_table=overlap_pivot,
    title='Overlap Coefficient Heatmap (Leiden: Structure vs. Sequence)',
    x_label='Sequence Clusters',
    y_label='Structure Clusters'
)

### Interpretation

The overlap coefficient should show me there are structural clusters that are subsets of sequence clusters. (or the other way around)

In [ ]:
# Compute statistics for each cluster for the different clustering methods

# statistics for HDBSCAN structure clusters
stats_per_cluster = evaluator.clustering.compute_cluster_statistics(
    df_cluster_labels=df_cluster_labels,
    cluster_col="cluster_id_struct_HDBSCAN",
    df_distances=df_euclidean_dist,
    distance_col="euclidean_dist_min_max_struct",
    df_scores=df_scores,
    score_col="Score_-log10_struct"
)
print("Cluster Statistics (HDBSCAN - structure):")
display(stats_per_cluster)

# statistics for HDBSCAN sequence clusters
stats_per_cluster = evaluator.clustering.compute_cluster_statistics(
    df_cluster_labels=df_cluster_labels,
    cluster_col="cluster_id_seq_HDBSCAN",
    df_distances=df_euclidean_dist,
    distance_col="euclidean_dist_min_max_seq",
    df_scores=df_scores,
    score_col="Score_-log10_seq"
)
print("Cluster Statistics (HDBSCAN - sequence):")
display(stats_per_cluster)

# statistics for Leiden structure clusters
stats_per_cluster = evaluator.clustering.compute_cluster_statistics(
    df_cluster_labels=df_cluster_labels,
    cluster_col="cluster_id_Score_-log10_struct_Leiden",
    df_distances=df_euclidean_dist,
    distance_col="euclidean_dist_min_max_struct",
    df_scores=df_scores,
    score_col="Score_-log10_struct"
)
print("Cluster Statistics (Leiden - structure):")
display(stats_per_cluster)

# statistics for Leiden sequence clusters
stats_per_cluster = evaluator.clustering.compute_cluster_statistics(
    df_cluster_labels=df_cluster_labels,
    cluster_col="cluster_id_Score_-log10_seq_Leiden",
    df_distances=df_euclidean_dist,
    distance_col="euclidean_dist_min_max_seq",
    df_scores=df_scores,
    score_col="Score_-log10_seq"
)
print("Cluster Statistics (Leiden - sequence):")
display(stats_per_cluster)

# next steps: 

1. create plots from:  

    jaccard index, cluster overlap coefficient for hdbscan and leiden for structure vs sequence based clustering  
    statistics for each cluster  

2. run evaluation with bigger dataset
    find pvalue then cluster

3. make sense of results